# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR$^2$ dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema, accessible at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. The metadata will provide information about the dataset, including its record sets, fields, and description.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset schema and metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Let's review the available record sets and their fields. All references will use the `@id` fields according to Croissant best practices.

In [ ]:
# List all record set @id's and their fields
print("Available Record Sets:")
for record_set in dataset.record_sets:
    print(f"  Record Set @id: {record_set['@id']}")
    print(f"    name: {record_set.get('name', '<no name>')}")
    print(f"    description: {record_set.get('description', '<no description>')}")
    # List fields for this record set
    if 'field' in record_set:
        print("    Fields:")
        for field in record_set['field']:
            print(f"      @id: {field['@id']}")
            print(f"        name: {field.get('name', '<no name>')}")
            print(f"        dataType: {field.get('dataType', '<no dataType>')}")
    else:
        print("    (No fields defined)")
    print("-")

## 3. Data Extraction
Extract and inspect the records from each record set, loading each into a pandas DataFrame using only their `@id` fields.

In [ ]:
# Find all record set @id's
record_set_ids = [record_set['@id'] for record_set in dataset.record_sets]
print(f"Record set @id's found: {record_set_ids}")

dataframes = {}
for rsid in record_set_ids:
    print(f"Loading records for record set {rsid}...")
    records = list(dataset.records(record_set=rsid))
    dataframes[rsid] = pd.DataFrame(records)
    print(f"Columns for {rsid}: {dataframes[rsid].columns.tolist()}")
    print(dataframes[rsid].head(2))

## 4. Exploratory Data Analysis (EDA)
Let's select one record set for demonstration—typically the main tabular record set containing patient and clinicopathological information. We'll reference it by its Croissant `@id`. We will demonstrate filtering by a numeric field, normalization, and grouping by another key attribute. All columns are referenced by their `@id`.

_Note: Please adapt `main_record_set_id`, `numeric_field_id`, and `group_field_id` as needed—see above for your dataset's specific IDs._

In [ ]:
# Select the main records set (replace with your actual record set @id from overview, typically the primary table)
if record_set_ids:
    main_record_set_id = record_set_ids[0]  # Use the first record set as example
else:
    main_record_set_id = None

df = dataframes[main_record_set_id]

# Display columns and example values to choose valid field @id's
print("Available columns:")
print(df.columns.tolist())
print(df.head(3))

# Let's find a likely numeric field and group field from the columns
numeric_field_id = None
for col in df.columns:
    # Heuristic for demo: if the column dtype is int or float, and not id
    if pd.api.types.is_numeric_dtype(df[col]) and 'id' not in col.lower():
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Fallback to the first column
    numeric_field_id = df.columns[0]

# Similarly, pick a likely group field (categorical, not id, not numeric)
group_field_id = None
for col in df.columns:
    if not pd.api.types.is_numeric_dtype(df[col]) and 'id' not in col.lower():
        group_field_id = col
        break

print(f"Numeric field chosen for EDA: {numeric_field_id}")
print(f"Group field chosen for grouping: {group_field_id}")

# EDA: Filtering, normalization, and grouping
if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
    filtered_df = df[df[numeric_field_id] > threshold] if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else df
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalization
    filtered_df = filtered_df.copy()  # to avoid SettingWithCopyWarning
    if pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Group by field if available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped means of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize the distribution of a numeric variable and (optionally) compare groups. All field and grouping references use their `@id`.

_Plots can be adjusted by selecting different field `@id`s as suited to your data._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of chosen numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id}")
plt.show()

# Boxplot by group field if available
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(10,4))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.xticks(rotation=45)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
Using the `mlcroissant` library, we've loaded the FAIR$^2$ colorectal cancer survivor dataset by referencing record sets and fields by their Croissant `@id`. We explored the available metadata, loaded tabular data, filtered and normalized numeric fields, grouped by relevant attributes, and produced visual summaries. 

You can extend this workflow by selecting other columns or adapting data processing to your specific analysis questions. Always use the field and record set `@id` as shown for reproducibility and interoperability with the Croissant schema.